# 21. 학습 데이터 세트 스캔 (Training Dataset Scan)

이 노트북은 모델 학습 직전, 이전 단계(18, 19, 20번)에서 생성된 데이터들이 의도대로 준비되었는지 최종 점검합니다.

**점검 대상:**
1. **가공 완료된 원본 데이터**: `train`, `val_tune`, `val_calib`, `test`
2. **학습용 서브셋 (19번)**: 언더배깅 분할 데이터 (`subset_0~9`)
3. **샘플링 검증셋 (20번)**: 1:100 가중치 샘플링 데이터

In [6]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from pathlib import Path
import config.train_config as cfg

def get_stats(df, name):
    pos = df[cfg.TARGET_COL].sum()
    neg = len(df) - pos
    return {
        'Dataset': name,
        'Total Rows': f"{len(df):,}",
        'Positive (Failure)': f"{pos:,}",
        'Negative (Normal)': f"{neg:,}",
        'Ratio (1:N)': f"1:{neg/pos:.1f}" if pos > 0 else "N/A"
    }

print("✅ 환경 준비 완료")

✅ 환경 준비 완료


## 1. 원본 가공 데이터 점검
Feature Engineering이 완료된 전체 데이터셋 현황입니다.

In [7]:
paths_raw = [
    ('Train (Full)', cfg.TRAIN_PATH),
    ('Val Tune (Full)', cfg.VAL_TUNE_PATH),
    ('Val Calib (Full)', cfg.VAL_CALIB_PATH),
    ('Test (Full)', cfg.TEST_PATH)
]

stats_list = []
for name, p in paths_raw:
    if Path(p).exists():
        df = pd.read_parquet(p, columns=[cfg.TARGET_COL])
        stats_list.append(get_stats(df, name))
    else:
        print(f"❌ [Missing] {name}: {p}")

pd.DataFrame(stats_list)

❌ [Missing] Train (Full): c:\Workspace\COIN\ML_HDD\data\split_group_stratified\train.parquet
❌ [Missing] Val Calib (Full): c:\Workspace\COIN\ML_HDD\data\split_group_stratified\val_calib_raw.parquet
❌ [Missing] Test (Full): c:\Workspace\COIN\ML_HDD\data\split_group_stratified\test_raw.parquet


,Dataset,Total Rows,Positive (Failure),Negative (Normal),Ratio (1:N)
0,Val Tune (Full),"7,902,193","5,536","7,896,657",1:1426.4


## 2. 학습용 서브셋 점검 (19번 결과)
10:1 비율로 언더샘플링된 학습용 서브셋입니다.

In [8]:
subset_path = Path(cfg.SUBSET_DIR)
subset_files = sorted(list(subset_path.glob("subset_*.parquet")))

if not subset_files:
    print(f"❌ [Missing] {cfg.SUBSET_DIR} 에 서브셋 파일이 없습니다.")
else:
    sub_stats = []
    for f in [subset_files[0], subset_files[-1]]:
        df = pd.read_parquet(f, columns=[cfg.TARGET_COL])
        sub_stats.append(get_stats(df, f.name))
    
    print(f"✅ 총 {len(subset_files)}개의 서브셋 감지됨")
    display(pd.DataFrame(sub_stats))

✅ 총 10개의 서브셋 감지됨


,Dataset,Total Rows,Positive (Failure),Negative (Normal),Ratio (1:N)
0,subset_0.parquet,"367,807","33,437","334,370",1:10.0
1,subset_9.parquet,"367,807","33,437","334,370",1:10.0


## 3. 샘플링 검증셋 점검 (20번 결과)
원본 Val Tune에서 전략적으로 추출된 고밀도 검증셋입니다.

In [9]:
val_sampled_path = Path(cfg.VAL_TUNE_SAMPLED_PATH)

if val_sampled_path.exists():
    df_val = pd.read_parquet(val_sampled_path)
    print(f"✅ 샘플링 검증셋 로드 완료: {val_sampled_path.name}")
    
    # 원본 Val Tune과 비교
    full_val_df = pd.read_parquet(cfg.VAL_TUNE_PATH, columns=[cfg.TARGET_COL])
    
    comparison = [
        get_stats(full_val_df, 'Val Tune (Full)'),
        get_stats(df_val, 'Val Tune (Sampled)')
    ]
    
    print("\n📊 원본 vs 샘플링 비교:")
    display(pd.DataFrame(comparison))
    
    print("\n🔍 샘플링 데이터 요약:")
    print(f"  - 총 행 수: {len(df_val):,}")
    print(f"  - 고장 개수: {df_val[cfg.TARGET_COL].sum():,}")
    display(df_val.head())
else:
    print(f"❌ [Missing] 샘플링 검증셋이 없습니다: {val_sampled_path}")

✅ 샘플링 검증셋 로드 완료: val_sampled.parquet

📊 원본 vs 샘플링 비교:


,Dataset,Total Rows,Positive (Failure),Negative (Normal),Ratio (1:N)
0,Val Tune (Full),"7,902,193","5,536","7,896,657",1:1426.4
1,Val Tune (Sampled),"559,136","5,536","553,600",1:100.0



🔍 샘플링 데이터 요약:
  - 총 행 수: 559,136
  - 고장 개수: 5,536


,serial_number,date,failure,s187_days_since_first,smart_184_raw,error_density_14d,smart_187_raw,smart_198_raw,total_seeks_28d_asfd,s187_28d_sum,...,s194_28d_std,s242_28d_dai,total_reads_7d_asfd,total_reads_7d_max,s194_14d_max,s194_28d_ewma,s194_14d_std,s190_28d_zscore,s190_28d_ewma,s190_28d_mean
0,Z305JH9T_1,2024-03-15,0,-1,0.0,0.0,0.0,0.0,15311003.0,0.0,...,0.875142,-4.357294e+06,6.615312e+08,109395064.0,30.0,28.484518,0.974961,1.836416,28.484518,28.392857
1,Z305FV8N_2,2023-07-02,0,-1,0.0,0.0,0.0,0.0,20973402.0,0.0,...,0.497347,6.092246e+05,9.307613e+08,121583536.0,25.0,24.531513,0.497245,-0.789889,24.531513,24.392857
2,Z304REB2_3,2022-10-26,0,-1,0.0,0.0,0.0,0.0,22447516.0,0.0,...,0.262265,6.161188e+06,7.644933e+08,71860016.0,28.0,27.923096,0.363137,0.272342,27.923096,27.928572
3,Z301QZWJ_1,2018-02-13,0,-1,0.0,0.0,0.0,0.0,2852197.0,0.0,...,0.792658,8.944430e+04,1.116676e+09,137649056.0,24.0,23.721251,0.000000,0.585725,23.721251,23.535715
4,S300YCYG_1,2017-08-29,0,-1,0.0,0.0,0.0,0.0,698012.0,0.0,...,0.503953,-4.945307e+04,7.763614e+08,166493728.0,29.0,28.589285,0.267261,0.850403,28.589285,28.571428


## 4. 최종 결론
모든 데이터셋의 'Positive' 개수가 원본과 일치하는지 확인하세요. 
특히 **원본 Val Tune**과 **샘플링 검증셋**의 고장 데이터 개수가 동일해야 정상입니다.